# Week 3-4: pandas and data manipulation
We clean a messy customer file step by step. Every step is a function in `mlkit`, so the same code runs in the project pipeline.

In [ ]:
import sys
sys.path.append("..")      # project root, so `import mlkit` works

In [ ]:
import pandas as pd
from scripts.make_sample_data import make_raw_customers
from mlkit import cleaning, features, validation
from mlkit.data_io import read_table, write_table
from mlkit.config import DATA_INTERIM, DATA_PROCESSED

## 1. Load and inspect
Always inspect first: shape, types, missing values, duplicates.

In [ ]:
raw = make_raw_customers()
print(raw.shape); print(raw.head(3).to_string()); raw.info()
print(raw.isna().sum()[lambda s: s > 0]); print("duplicates:", raw.duplicated().sum())

## 2. Select and filter
`loc` uses labels, `iloc` uses positions, and boolean masks filter rows.

In [ ]:
df = cleaning.standardize_columns(raw)
print(df.columns.tolist())
print(df.loc[0:2, ["tenure_months", "plan"]])
print(df.iloc[0:3, 1:3])
print(df[(df.support_tickets >= 4) & (df.tenure_months < 12)].shape)
print(df.query("monthly_charges > 100").shape)

## 3. GroupBy: summarize by category

In [ ]:
df = cleaning.clean_text(df, ["plan"])
print(df.plan.value_counts())
print(df.groupby("plan").agg(customers=("customer_id", "count"), churn_rate=("churn", "mean"), avg_charges=("monthly_charges", "mean")).round(3))

## 4. Merge: join a second table
Left join keeps every customer even if the lookup has no match.

In [ ]:
prices = pd.DataFrame({"plan": ["basic", "standard"], "list_price": [50, 80]})   # premium is missing on purpose
merged = df.merge(prices, on="plan", how="left")
print(merged.groupby("plan").list_price.first())

## 5. Dates
Turn tenure into a start date, then group by month.

In [ ]:
df["start_date"] = pd.Timestamp("2026-10-01") - pd.to_timedelta(df.tenure_months * 30, unit="D")
df["start_month"] = df.start_date.dt.to_period("M")
print(df.groupby("start_month").size().tail(3))

## 6. Clean with the toolkit
**Reusable:** these five functions are the cleaning stage of the project pipeline.

In [ ]:
df = cleaning.standardize_columns(raw)
df = cleaning.clean_text(df, ["plan"])
df = cleaning.drop_duplicates(df)
print("rows after dedupe:", len(df), "| missing charges:", df.monthly_charges.isna().sum(), "| max charges:", df.monthly_charges.max())
df = cleaning.fill_missing(df, ["monthly_charges"])
df = cleaning.clip_outliers(df, "monthly_charges")
print("missing charges:", df.monthly_charges.isna().sum(), "| max charges:", round(df.monthly_charges.max(), 2))

## 7. Features, validation, save
**Reusable:** `add_features` is shared by training and serving. `validate` stops bad data before it reaches a model. Parquet keeps types and is smaller than CSV.

In [ ]:
df = features.add_features(df)
SCHEMA = {"tenure_months": {"min": 1}, "monthly_charges": {"min": 0},
          "plan": {"allowed": {"basic", "standard", "premium"}}, "churn": {"allowed": {0, 1}}}
validation.validate(df, SCHEMA)
print(validation.data_profile(df))
write_table(df, DATA_INTERIM / "demo.parquet")
print(read_table(DATA_INTERIM / "demo.parquet").dtypes.to_dict())

## 8. What happens when data is bad?

In [ ]:
bad = df.copy(); bad.loc[0, "plan"] = "gold"; bad.loc[1, "tenure_months"] = -5
try:
    validation.validate(bad, SCHEMA)
except validation.ValidationError as e:
    print("blocked:", e)

## Try it yourself
1. Which plan has the highest churn rate? Which has the highest average charges?
2. Fill `monthly_charges` with the **mean** instead of the median and compare.
3. Add a rule to `SCHEMA` that `logins_per_week` is at most 50, then break it on purpose.